### Title:- Design RNN or its variant including LSTM or GRU
a) Select a suitable time series dataset. E.g - Predict sentiments based on product reviews.

b) Apply for prediction

#### Import Required Libraries

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

#### Load Dataset

In [2]:
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


#### Preprocessing (Padding Sequences)

In [3]:
max_len = 200

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

print(X_train.shape)

(25000, 200)


#### Build LSTM Model

In [4]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len))
model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

c:\Users\Prafull Satle\tf-env\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#### Train Model

In [5]:
early_stop = EarlyStopping(monitor='val_loss', patience=2)

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 96s 269ms/step - accuracy: 0.7036 - loss: 0.5431 - val_accuracy: 0.8434 - val_loss: 0.3686
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 251ms/step - accuracy: 0.8802 - loss: 0.3039 - val_accuracy: 0.8574 - val_loss: 0.3200
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 79s 254ms/step - accuracy: 0.9313 - loss: 0.1910 - val_accuracy: 0.8606 - val_loss: 0.3497
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 76s 244ms/step - accuracy: 0.9539 - loss: 0.1323 - val_accuracy: 0.8454 - val_loss: 0.4163


#### Evaluate Model

In [6]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 42s 53ms/step - accuracy: 0.8462 - loss: 0.4241
Test Accuracy: 0.8490399718284607


#### Prediction Example

In [7]:
sample = X_test[0].reshape(1, -1)
prediction = model.predict(sample)

print("Predicted Sentiment:", "Positive" if prediction[0][0] > 0.5 else "Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 706ms/step
Predicted Sentiment: Positive


#### Prediction on Unseen data


In [8]:
from tensorflow.keras.datasets import imdb

word_index = imdb.get_word_index()

In [9]:
def encode_review(text):
    words = text.lower().split()
    encoded = []

    for word in words:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word] + 3)  # +3 because IMDB reserves indices
        else:
            encoded.append(2)  # unknown word

    return encoded

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def preprocess_review(text):
    encoded = encode_review(text)
    padded = pad_sequences([encoded], maxlen=max_len)
    return padded

In [11]:
review = "This movie was absolutely amazing, I loved it"

processed_review = preprocess_review(review)

prediction = model.predict(processed_review)

print("Prediction Score:", prediction[0][0])

if prediction[0][0] > 0.5:
    print("Sentiment: Positive 😊")
else:
    print("Sentiment: Negative 😞")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Prediction Score: 0.9065834
Sentiment: Positive 😊


In [12]:
review1 = "The movie was fantastic and full of emotions"
review2 = "Worst movie ever, waste of time"

for r in [review1, review2]:
    pred = model.predict(preprocess_review(r))
    print(r)
    print("Sentiment:", "Positive" if pred[0][0] > 0.5 else "Negative")
    print()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
The movie was fantastic and full of emotions
Sentiment: Positive

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
Worst movie ever, waste of time
Sentiment: Negative

